**Import Modules**

In [13]:
import numpy as np
import pandas as pd
import joblib as jb

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

**Import Dataset**

In [3]:
# Current folder: KG_Modeling
BASE_DIR = Path.cwd().resolve()

# Locate the sibling folders
PROJECT_DIR = BASE_DIR.parent
ARTIFACTS_DIR = PROJECT_DIR / "KG_Loan_Artifacts"

# Import the CSV dataset
df = pd.read_csv(ARTIFACTS_DIR / "KG_Loan_Dataset.csv")

df.head()

,Female,Married,Dependents,Graduate,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Term,Credit,Approved,Rural,Semiurban,Urban
0,0,0,0,1,0,5849,0.0,128.0,360,1,1,0,0,1
1,0,1,1,1,0,4583,1508.0,128.0,360,1,0,1,0,0
2,0,1,0,1,1,3000,0.0,66.0,360,1,1,0,0,1
3,0,1,0,0,0,2583,2358.0,120.0,360,1,1,0,0,1
4,0,0,0,1,0,6000,0.0,141.0,360,1,1,0,0,1


In [5]:
df.columns

Index(['Female', 'Married', 'Dependents', 'Graduate', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Term', 'Credit',
       'Approved', 'Rural', 'Semiurban', 'Urban'],
      dtype='object')

In [9]:
scale_colum = ['Dependents', 'ApplicantIncome', 
        'CoapplicantIncome', 'LoanAmount', 'Term']

Apply scaling to numeric columns

In [10]:
scaler_obj = StandardScaler()

df[scale_colum] = scaler_obj.fit_transform(df[scale_colum])
df.head()

,Female,Married,Dependents,Graduate,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Term,Credit,Approved,Rural,Semiurban,Urban
0,0,0,-0.737806,1,0,0.072991,-0.554487,-0.211241,0.273231,1,1,0,0,1
1,0,1,0.253470,1,0,-0.134412,-0.038732,-0.211241,0.273231,1,0,1,0,0
2,0,1,-0.737806,1,1,-0.393747,-0.554487,-0.948996,0.273231,1,1,0,0,1
3,0,1,-0.737806,0,0,-0.462062,0.251980,-0.306435,0.273231,1,1,0,0,1
4,0,0,-0.737806,1,0,0.097728,-0.554487,-0.056551,0.273231,1,1,0,0,1


Save Scaler object to scale down future input

In [11]:
# Locate the artifacts folder relative to KG_Modeling
BASE_DIR = Path.cwd().resolve()
ARTIFACTS_DIR = BASE_DIR.parent / "KG_Loan_Artifacts"

# Save the already-fitted scaler
jb.dump(scaler_obj, ARTIFACTS_DIR / "KG_Loan_Scaler.pkl")

['C:\\Users\\KRISHNA\\OneDrive\\Desktop\\Krishna\\Programming\\KG_Loan_Approval\\KG_Loan_Artifacts\\KG_Loan_Scaler.pkl']

Train-Test split data

In [12]:
X = df.drop('Approved', axis = 1)
Y = df['Approved']

X_train, X_test, y_train, y_test = train_test_split( X, Y, test_size=0.2, random_state=42, stratify=Y)

Model:

first model with all features, then model with removing one feature to check if performance changes

In [16]:
models = []
results = []

features = list(X_train.columns)

# Model 0: All features
experiments = [("model_0", features, "None")]

# Remaining models: Remove one feature at a time
for feature in features:
    remaining = [col for col in features if col != feature]
    experiments.append(
        (f"model_{len(experiments)}", remaining, feature)
    )

for name, cols, removed in experiments:
    model_obj = LogisticRegression(max_iter=1000)
    model_obj.fit(X_train[cols], y_train)

    pred = model_obj.predict(X_test[cols])
    prob = model_obj.predict_proba(X_test[cols])[:, 1]

    results.append({
        "Model": name,
        "Removed_Feature": removed,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, pred, pos_label=1, zero_division=0),
        "F1_Score": f1_score(y_test, pred, pos_label=1, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })

    models.append(model_obj)

evaluation_df = pd.DataFrame(results)

evaluation_df.round(4)

,Model,Removed_Feature,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,model_0,None,0.8618,0.8400,0.9882,0.9081,0.8486
1,model_1,Female,0.8618,0.8400,0.9882,0.9081,0.8551
2,model_2,Married,0.8618,0.8400,0.9882,0.9081,0.8464
3,model_3,Dependents,0.8618,0.8400,0.9882,0.9081,0.8511
4,model_4,Graduate,0.8618,0.8400,0.9882,0.9081,0.8502
5,model_5,Self_Employed,0.8618,0.8400,0.9882,0.9081,0.8576
6,model_6,ApplicantIncome,0.8618,0.8400,0.9882,0.9081,0.8492
7,model_7,CoapplicantIncome,0.8537,0.8317,0.9882,0.9032,0.8576
8,model_8,LoanAmount,0.8618,0.8400,0.9882,0.9081,0.8458
9,model_9,Term,0.8618,0.8400,0.9882,0.9081,0.8486


In [ ]:
model_0 = models[0]

model_0.predict(X_test)
model_0.predict_proba(X_test)[:, 1]
print("Accuracy", accuracy_score(y_test, pred),)
print("Precision", precision_score(y_test, pred, pos_label=1, zero_division=0),)
print("Recall", recall_score(y_test, pred, pos_label=1, zero_division=0),)
print("F1_Score", f1_score(y_test, pred, pos_label=1, zero_division=0),)
print("ROC_AUC", roc_auc_score(y_test, prob))
model_0.feature_names_in_

Accuracy 0.8617886178861789
Precision 0.84
Recall 0.9882352941176471
F1_Score 0.9081081081081082
ROC_AUC 0.8486068111455107


array(['Female', 'Married', 'Dependents', 'Graduate', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Term',
       'Credit', 'Rural', 'Semiurban', 'Urban'], dtype=object)

In [21]:
print("Model:", model_0)
print("Parameters:", model_0.get_params())

print("Number of features:", model_0.n_features_in_)
print("Feature names:", model_0.feature_names_in_)

print("Target classes:", model_0.classes_)
print("Coefficients:", model_0.coef_)
print("Intercept:", model_0.intercept_)
print("Training iterations:", model_0.n_iter_)

Model: LogisticRegression(max_iter=1000)
Parameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 1000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
Number of features: 13
Feature names: ['Female' 'Married' 'Dependents' 'Graduate' 'Self_Employed'
 'ApplicantIncome' 'CoapplicantIncome' 'LoanAmount' 'Term' 'Credit'
 'Rural' 'Semiurban' 'Urban']
Target classes: [0 1]
Coefficients: [[ 1.45860604e-01  4.87902419e-01  9.85079640e-02  3.80286034e-01
  -1.97685609e-01 -7.74367575e-03 -1.46309068e-01 -3.36978078e-02
  -5.89008700e-05  3.13649858e+00 -2.86434887e-01  4.34939621e-01
  -1.52108142e-01]]
Intercept: [-2.50638053]
Training iterations: [26]


Save model

In [22]:
jb.dump(model_0, ARTIFACTS_DIR / "KG_Loan_Predictor.pkl")

['C:\\Users\\KRISHNA\\OneDrive\\Desktop\\Krishna\\Programming\\KG_Loan_Approval\\KG_Loan_Artifacts\\KG_Loan_Predictor.pkl']